# 03 — Project Any NIfTI Map onto the Brain Surface

This notebook projects any 3D NIfTI stat/contrast/FC map onto the fsaverage inflated surface.
Useful for visualising GLM results, seed-based FC maps, or any volumetric brain map.

**Dependencies:** `nilearn`, `nibabel`, `numpy`, `matplotlib`

## 0 — Configuration

Set the path to your 3D NIfTI and the colormap range.

In [ ]:
# ── Edit these ─────────────────────────────────────────────────────────
NIFTI_PATH  = '/path/to/your/stat_map.nii.gz'   # <-- 3D NIfTI
TITLE       = 'My Brain Map'
CBAR_LABEL  = 'z-score'
VMIN        = -3
VMAX        =  3
CMAP        = 'RdBu_r'
# ───────────────────────────────────────────────────────────────────────

## 1 — Imports & Helper

In [ ]:
import numpy as np
from nilearn import plotting, image, surface, datasets
import matplotlib.pyplot as plt
from datetime import datetime

now = datetime.now()

In [ ]:
def nifti_to_surface(nifti_image, nan_threshold=0.0005):
    """Project a volumetric NIfTI onto fsaverage left/right pial surfaces."""
    fsaverage = datasets.fetch_surf_fsaverage()
    tl = surface.vol_to_surf(nifti_image, fsaverage.pial_left,
                              interpolation='linear', radius=3.0, n_samples=20)
    tr = surface.vol_to_surf(nifti_image, fsaverage.pial_right,
                              interpolation='linear', radius=3.0, n_samples=20)
    tl[np.abs(tl) <= nan_threshold] = np.nan
    tr[np.abs(tr) <= nan_threshold] = np.nan
    return tl, tr


def surface_plot(texture_left, texture_right, title='', colorbarlabel='',
                 vmin=-1, vmax=1, cmap='RdBu_r',
                 views=('lateral', 'medial'), save=False):
    """Four-panel inflated surface plot."""
    fsaverage = datasets.fetch_surf_fsaverage()
    fig, axes = plt.subplots(1, 4, figsize=(15, 3), subplot_kw={'projection': '3d'})
    fig.suptitle(title, fontsize=12, x=0.45)

    for ax, (view, hemi) in zip(axes, [(views[0], 'left'), (views[0], 'right'),
                                        (views[1], 'left'), (views[1], 'right')]):
        texture = texture_left if hemi == 'left' else texture_right
        plotting.plot_surf_stat_map(
            fsaverage.infl_left if hemi == 'left' else fsaverage.infl_right,
            texture, hemi=hemi, view=view, colorbar=False,
            bg_map=fsaverage.sulc_left if hemi == 'left' else fsaverage.sulc_right,
            cmap=cmap, axes=ax, vmax=vmax, vmin=vmin, bg_on_data=True, darkness=None
        )

    fig.subplots_adjust(top=1, right=0.85, wspace=0, hspace=0, left=0.05, bottom=0)
    cbar_ax = fig.add_axes([0.95, 0.05, 0.02, 0.8])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    plt.colorbar(sm, cax=cbar_ax).set_label(colorbarlabel, rotation=270, labelpad=15)

    if save:
        fname = str(now).replace(' ', '_').replace(':', '_').split('.')[0] + f'_{title}.png'
        plt.savefig(fname, transparent=True, dpi=300)
    plt.show()

## 2 — Load & Project

In [ ]:
nii = image.load_img(NIFTI_PATH)
print(f'Shape     : {nii.shape}')
print(f'Voxel size: {nii.header.get_zooms()}')

texture_left, texture_right = nifti_to_surface(nii)
surface_plot(texture_left, texture_right,
             title=TITLE, colorbarlabel=CBAR_LABEL,
             vmin=VMIN, vmax=VMAX, cmap=CMAP, save=False)

## Tips

- If your map is in a different space (e.g. 1 mm MNI), resample first:  
  `nii = image.resample_img(nii, target_affine=np.eye(3)*2)` (or use `resample_to_img`)
- Change `nan_threshold` in `nifti_to_surface` if too much / too little of the map shows up
- `save=True` saves a PNG in the current directory
- Swap `cmap='RdBu_r'` for `'hot'`, `'cold_hot'`, `'bwr'`, etc.